# Model Bundle Workflow Example

This notebook demonstrates the model bundle workflow for saving and loading trained gap-filling models with all necessary metadata and statistics.

## Why Use Model Bundles?

A model bundle packages together:
- The trained TensorFlow/Keras model
- Standardization statistics (mean, stdev)
- Training metadata (region, dates, configuration)
- Training history

This ensures reproducible gap-filling workflows by keeping all artifacts together.

## 1. Training and Saving a Model Bundle

After training your model (see `2-U-Net_Fit.ipynb`), save it as a bundle:

In [ ]:
import mindthegap as mtg

# After training (see 2-U-Net_Fit.ipynb for full training code):
# model = mtg.UNet(input_shape=(40, 56, 11))
# history = model.fit(train_dataset, validation_data=val_dataset, epochs=50)
# ds_out, stats = mtg.build_standardized_lazy(...)

# Save the complete bundle
bundle = mtg.save_model_bundle(
    model=model,
    bundle_path="models/arabsea_2015_bundle",
    stats=stats,
    metadata={
        "region": "Arabian Sea",
        "train_year": 2015,
        "train_range": 3,
        "patch_size": (40, 56),
        "n_channels": 11,
        "zarr_source": "gcs://nmfs_odp_nwfsc/CB/mind_the_chl_gap/IO.zarr",
        "notes": "U-Net with 3 encoder/decoder levels, trained on 2015-2017 data"
    },
    history=history,
    overwrite=True
)

print("✓ Bundle saved!")
print(bundle.info())

## 2. Loading a Model Bundle

Load the bundle to make predictions or continue evaluation:

In [ ]:
import mindthegap as mtg

# Load the bundle
bundle = mtg.load_model_bundle("models/arabsea_2015_bundle")

# Access components
model = bundle.model
stats = bundle.stats
metadata = bundle.metadata

print(bundle.info())

## 3. Using the Bundle for Predictions

Use the loaded model and stats for gap-filling:

In [ ]:
import numpy as np
import xarray as xr

# Load and prepare test data (example)
# zarr_ds = xr.open_dataset("gcs://...", engine="zarr", ...)
# ds_out, _ = mtg.build_standardized_lazy(zarr_ds, stats=stats, ...)

# Make predictions
# X_test = ...  # Prepare test patches
# predictions = model.predict(X_test)

# Unstandardize predictions using stats from bundle
# unstd_predictions = mtg.unstdize(predictions, stats['CHL'])

print("Using stats from bundle:")
print(f"  CHL mean: {stats['CHL'][0]:.4f}")
print(f"  CHL std:  {stats['CHL'][1]:.4f}")

## 4. Comparing Multiple Model Bundles

List all available bundles to compare models:

In [ ]:
import os
from pathlib import Path
import mindthegap as mtg

models_dir = Path("models")

# Find all bundles (directories containing model.keras)
for bundle_dir in models_dir.rglob("model.keras"):
    bundle_path = bundle_dir.parent
    print(f"\n📦 {bundle_path.name}")
    
    try:
        bundle = mtg.load_model_bundle(bundle_path)
        
        # Show key info
        if 'region' in bundle.metadata:
            print(f"   Region: {bundle.metadata['region']}")
        if 'train_year' in bundle.metadata:
            print(f"   Train year: {bundle.metadata['train_year']}")
        
        # Show final validation loss
        if bundle.history and 'val_loss' in bundle.history:
            final_val = bundle.history['val_loss'][-1]
            print(f"   Final val_loss: {final_val:.6f}")
    except Exception as e:
        print(f"   Error loading: {e}")

## 5. Command-Line Interface

You can also manage bundles from the command line:

```bash
# List all bundles
python -m mindthegap.cli list models/

# Show bundle info
python -m mindthegap.cli info models/arabsea_2015_bundle

# Export stats to JSON
python -m mindthegap.cli export-stats models/arabsea_2015_bundle --output my_stats.json
```

## Bundle Directory Structure

A model bundle has this structure:

```
models/arabsea_2015_bundle/
├── model.keras         # TensorFlow model
├── stats.json          # Standardization statistics
├── metadata.json       # Configuration and provenance
└── history.json        # Training history (optional)
```

All files are human-readable JSON (except the model), making it easy to inspect and version control metadata.

## Modified Training Workflow

Here's how to modify the training notebook (2-U-Net_Fit.ipynb) to use bundles:

**Old way:**
```python
# Save model only
model_path = f'{model_dir}/{zarr_label}/{model_name}.keras'
model.save(model_path)
```

**New way with bundles:**
```python
# Save complete bundle
bundle = mtg.save_model_bundle(
    model=model,
    bundle_path=f'{model_dir}/{zarr_label}_bundle',
    stats=stats,
    metadata={'region': 'Arabian Sea', 'train_year': 2015},
    history=history
)
```

**Loading for evaluation:**
```python
# Load complete bundle instead of just model
bundle = mtg.load_model_bundle(f'{model_dir}/{zarr_label}_bundle')
model = bundle.model
stats = bundle.stats
```